In [0]:
from pyspark.sql.functions import col, count, when

def run_null_checks(df, config):
    """
    Checks the number of NULL values in each column specified in the config. Returns a list of results for each column.
    """
    results = []
    for column, threshold in config["null_checks"].items():
        # Count number of NULLs in the current column
        null_count = df.filter(col(column).isNull()).count()
        status = "Pass" if null_count <= threshold else "Fail"
        results.append({
            "column_name": column,
            "check_name": "null_check",
            "metric_type": "count",
            "metric_value": null_count,
            "threshold": threshold,
            "status": status
        })
    return results


In [0]:
from pyspark.sql.functions import col, to_timestamp

def run_format_checks(df, config):
    """
    Checks format validity of columns based on regex, expected datatype, and timestamp parsing.
    Returns a list of results for each rule.
    """
    results = []
    format_section = config.get("format_checks", {})

    # ===============================
    # REGEX CHECKS
    # ===============================
    regex_rules = format_section.get("regex", {})
    for column, rules in regex_rules.items():
        pattern = rules.get("pattern")
        threshold = rules.get("threshold", 0)
        # Count non-null values NOT matching regex
        fail_count = df.filter(
            col(column).isNotNull() &
            ~col(column).rlike(pattern)
        ).count()
        status = "Pass" if fail_count <= threshold else "Fail"
        results.append({
            "column_name": column,
            "check_name": "regex_check",
            "metric_type": "count",
            "metric_value": fail_count,
            "threshold": threshold,
            "status": status
        })

    # ===============================
    # DATATYPE CHECKS
    # ===============================
    datatype_rules = format_section.get("datatype_check", {})
    for column, rules in datatype_rules.items():
        expected_type = rules.get("expected").lower()
        threshold = rules.get("threshold", 0)
        actual_type = df.schema[column].dataType.simpleString().lower()
        fail_count = 0 if expected_type in actual_type else 1
        status = "Pass" if fail_count <= threshold else "Fail"
        results.append({
            "column_name": column,
            "check_name": "datatype_check",
            "metric_type": "boolean",
            "metric_value": fail_count,
            "threshold": threshold,
            "status": status
        })

    # ===============================
    # TIMESTAMP FORMAT CHECKS
    # ===============================
    timestamp_rules = format_section.get("timestamp_check", {})
    for column, rules in timestamp_rules.items():
        fmt = rules.get("format")
        threshold = rules.get("threshold", 0)
        # Count cases where timestamp cannot be parsed
        fail_count = df.filter(
            col(column).isNotNull() &
            to_timestamp(col(column), fmt).isNull()
        ).count()
        status = "Pass" if fail_count <= threshold else "Fail"
        results.append({
            "column_name": column,
            "check_name": "timestamp_format_check",
            "metric_type": "count",
            "metric_value": fail_count,
            "threshold": threshold,
            "status": status
        })
    return results


In [0]:
def run_range_checks(df, config):
    """
    Checks if column values are within [min, max] inclusive for each configured rule.
    Returns a list of results for each column.
    """
    results = []
    for column, rules in config["range_checks"].items():
        # Count rows NOT within the valid range
        fail_count = df.filter(
            ~col(column).between(rules["min"], rules["max"])
        ).count()
        status = "Pass" if fail_count <= rules["threshold"] else "Fail"
        results.append({
            "column_name": column,
            "check_name": "range_check",
            "metric_type": "count",
            "metric_value": fail_count,
            "threshold": rules["threshold"],
            "status": status
        })
    return results


In [0]:
def run_date_range_checks(df, config):
    """
    Checks if date/time column is within [min, max] for each configured rule. Returns list of results.
    """
    results = []
    for column, rules in config["date_range_checks"].items():
        # Count rows whose dates fall outside the valid range
        fail_count = df.filter(
            ~col(column).between(rules["min"], rules["max"])
        ).count()
        status = "Pass" if fail_count <= rules["threshold"] else "Fail"
        results.append({
            "column_name": column,
            "check_name": "date_range_check",
            "metric_type": "count",
            "metric_value": fail_count,
            "threshold": rules["threshold"],
            "status": status
        })
    return results


In [0]:
def run_enum_checks(df, config):
    """
    Checks if column values only use allowed (enumerated) values. Returns a list of results.
    """
    results = []
    for column, rules in config["enum_checks"].items():
        # Count values not in the list of allowed values
        fail_count = df.filter(
            ~col(column).isin(rules["allowed_values"])
        ).count()
        status = "Pass" if fail_count <= rules["threshold"] else "Fail"
        results.append({
            "column_name": column,
            "check_name": "enum_check",
            "metric_type": "count",
            "metric_value": fail_count,
            "threshold": rules["threshold"],
            "status": status
        })
    return results


In [0]:
from pyspark.sql.functions import col

def run_pk_check(df, config):
    """
    Checks composite or single-column primary key uniqueness (excludes rows with null PKs). Returns a single result dict.
    """
    pk_config = config["primary_key"]
    pk_columns = pk_config.get("column", [])
    threshold = pk_config["threshold"]
    # Accept string or list for PK columns
    if isinstance(pk_columns, str):
        pk_columns = [pk_columns]
    column_name_str = ", ".join(pk_columns)
    # Remove rows where any PK column is null
    non_null_df = df
    for c in pk_columns:
        non_null_df = non_null_df.filter(col(c).isNotNull())
    # Count duplicate keys
    fail_count = (
        non_null_df
        .groupBy(*pk_columns)
        .count()
        .filter(col("count") > 1)
        .count()
    )
    status = "Pass" if fail_count <= threshold else "Fail"
    return [{
        "column_name": column_name_str,
        "check_name": "primary_key_check",
        "metric_type": "count",
        "metric_value": fail_count,
        "threshold": threshold,
        "status": status
    }]
